# VNA Stage B5 — measure only, T4×2
Set the immutable `--commit` pin after publishing this implementation. This
notebook includes neither Stage A nor B6. Install pinned upstream dependencies,
run the full required suite, and smoke two tasks of each registered arm before
budget approval. GPU0 runs ImageNet BP then CBP; GPU1 runs regression BP then
all five registered CBP replacement rates, then MNIST BP and CBP. A synthetic
smoke is explicitly flagged invalid; it never establishes the baseline gate.
Hourly archives retain task-boundary resume state and RNG. Attach the prior
archive to resume. MNIST uses the original upstream pickle builder; attached
raw data is preferred, otherwise its loader downloads MNIST. Missing ImageNet
records `skipped: data unavailable` and continues. No resets or VNA corrections
are applied to the training models.


In [ ]:
from pathlib import Path
REPO_COMMIT = "0effb30fa565afe56b975fb7bb5364b21c210133"
GPU_HOURS_BUDGET = 8
ALLOW_OVER_BUDGET = False
IMAGENET32_DIR = None  # directory containing per-class <class_id>.npy
MNIST_DIR = None       # directory containing mnist_ or torchvision MNIST/raw
MU_REL_DEFAULT = 1e-2
MU_REL_DEFAULT_SOURCE = "fallback:1e-2"
DISK_CAP_BYTES = 5_000_000_000


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
REFERENCE_URL = "https://github.com/duyh80456-code/One-Shot-TAS-CCIL.git"
REFERENCE = Path("/kaggle/working/One-Shot-TAS-CCIL")
GROMO_URL = "https://github.com/growingnet/gromo.git"
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
GROMO = Path("/kaggle/working/gromo")
RUN_ROOT = Path("/kaggle/working/vna_stage_b5")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if branch == REPO_COMMIT:
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", REPO_COMMIT], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", REPO_COMMIT], check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO, REFERENCE, GROMO):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
private_clone(REFERENCE_URL, REFERENCE, "ccil-residual-capacity")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                GROMO_URL, str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1",
                "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach",
                GROMO_COMMIT], check=True)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(GROMO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
GROMO_SRC = GROMO / "src"
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), str(GROMO_SRC), str(REFERENCE), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO,
               env={**test_env, "CUDA_VISIBLE_DEVICES": ""}, check=True)
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")

# Read the attached Stage A default before constructing any B5 command.
import json
stage_a_summaries = sorted(Path("/kaggle/input").glob("**/stageA/summary.json"))
if stage_a_summaries:
    MU_REL_DEFAULT_SOURCE = str(stage_a_summaries[0])
    summary = json.loads(stage_a_summaries[0].read_text())
    MU_REL_DEFAULT = summary["mu_rel_default"]
else:
    MU_REL_DEFAULT = 1e-2
    MU_REL_DEFAULT_SOURCE = "fallback:1e-2"
    print("WARNING: no attached stageA/summary.json; using MU_REL_DEFAULT=1e-2")
print("B5 mu_rel_default:", MU_REL_DEFAULT, "source:", MU_REL_DEFAULT_SOURCE)
# Prefer attached input data. ImageNet absence is an explicit skip.
if IMAGENET32_DIR is None:
    candidates = sorted({p.parent for p in Path("/kaggle/input").rglob("0.npy") if (p.parent / "1.npy").exists()})
    IMAGENET32_DIR = candidates[0] if candidates else None
if MNIST_DIR is None:
    pickles = sorted(Path("/kaggle/input").rglob("mnist_"))
    raw = sorted(Path("/kaggle/input").rglob("train-images-idx3-ubyte"))
    MNIST_DIR = pickles[0].parent if pickles else (raw[0].parents[2] if raw else None)
print("MNIST:", MNIST_DIR, "ImageNet-32:", IMAGENET32_DIR)


In [ ]:
from experiments.vna_kaggle import archive_outputs as _archive_outputs
from experiments.stage_b5 import restore_archives
from experiments.stage_b5 import RATES, finalize
import threading
STAGE_B = RUN_ROOT / "outputs" / "stageB"
STAGE_B.mkdir(parents=True, exist_ok=True)
restore_archives(Path("/kaggle/input"), STAGE_B)
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"}
archive_lock = threading.Lock()
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES, folders=("outputs/stageB",), exclude_parts=("data",))
archive_stop = threading.Event()
archive_errors = []
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
def command(benchmark, arm, gpu, out, *, smoke=False, rate=None):
    cmd = [sys.executable, "-u", "-m", "experiments.stage_b5", benchmark, "--arm", arm,
           "--device", f"cuda:{gpu}", "--out", str(out), "--mu-rel-default", str(MU_REL_DEFAULT),
           "--mu-rel-default-source", MU_REL_DEFAULT_SOURCE]
    if smoke: cmd += ["--smoke"]
    else: cmd += ["--resume"]
    if rate is not None: cmd += ["--replacement-rate", str(rate)]
    data = IMAGENET32_DIR if benchmark == "imagenet" else MNIST_DIR if benchmark == "mnist" else None
    if data: cmd += ["--data-dir", str(data)]
    return cmd
CONFIGS = [("regression", "bp", 1, None)] + [("regression", "cbp", 1, r) for r in RATES]
CONFIGS += [("mnist", "bp", 1, None), ("mnist", "cbp", 1, None)]
if IMAGENET32_DIR is not None:
    CONFIGS += [("imagenet", "bp", 0, None), ("imagenet", "cbp", 0, None)]
else:
    subprocess.run(command("imagenet", "bp", 0, STAGE_B), cwd=REPO, env=ENV, check=True)


In [ ]:
# Smoke native architectures and real data; measure the full diagnostics cost.
# MNIST's tiny rank-log fixture is used only in smoke and CPU tests.
SMOKE = RUN_ROOT / "smoke" / "outputs" / "stageB"
try:
    smoke_seconds = 0.
    projected = 0.
    for benchmark, arm, gpu, rate in CONFIGS:
        start = time.monotonic()
        subprocess.run(command(benchmark, arm, gpu, SMOKE, smoke=True, rate=rate), cwd=REPO, env=ENV, check=True)
        elapsed = time.monotonic()-start; smoke_seconds += elapsed
        arm_name = arm if rate is None else f"cbp_rr_{rate:g}"
        summary = json.loads((SMOKE / "b5" / benchmark / arm_name / "summary.json").read_text())
        full_steps = {"regression": 1000000, "mnist": 12000000, "imagenet": 600000}[benchmark]
        # Training and diagnostics have different scaling: project training
        # per step, then the registered 16/18 measurement points separately.
        # Short online smoke tasks measure pure partial windows at midpoints.
        train_seconds = summary["training_seconds"] * full_steps / summary["steps_run"]
        point_seconds = max((p["seconds"] for p in summary["measurement_point_seconds"]), default=0.)
        point_count = 16 if benchmark == "regression" else 18
        hours = (train_seconds + point_count*point_seconds) / 3600
        projected += hours
        print(benchmark, arm_name, f"{hours:.3f} projected GPU-hours", flush=True)
    total = projected + smoke_seconds/3600
    print(f"Total including smoke: {total:.3f} GPU-hours; budget: {GPU_HOURS_BUDGET}")
    (STAGE_B / "cost_projection.json").write_text(json.dumps({"projected_gpu_hours": projected,
        "smoke_gpu_hours": smoke_seconds/3600, "total_gpu_hours": total}, indent=2))
    archive_outputs()
    if total > GPU_HOURS_BUDGET and not ALLOW_OVER_BUDGET:
        raise RuntimeError("Projected B5 cost exceeds GPU_HOURS_BUDGET; review before setting ALLOW_OVER_BUDGET")
except Exception:
    archive_stop.set(); archiver.join(timeout=10); archive_outputs(); raise


In [ ]:
from concurrent.futures import ThreadPoolExecutor
def worker(gpu):
    for benchmark, arm, configured_gpu, rate in CONFIGS:
        if configured_gpu != gpu: continue
        arm_name = arm if rate is None else f"cbp_rr_{rate:g}"
        with (RUN_ROOT / f"{benchmark}_{arm_name}_gpu{gpu}.log").open("a", buffering=1) as log:
            subprocess.run(command(benchmark, arm, gpu, STAGE_B, rate=rate), cwd=REPO, env=ENV,
                           stdout=log, stderr=subprocess.STDOUT, check=True)
        print("Finished", benchmark, arm_name, "GPU", gpu, flush=True)
        archive_outputs()
try:
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = [pool.submit(worker, gpu) for gpu in (0, 1)]
        for future in futures: future.result()
finally:
    archive_stop.set(); archiver.join(timeout=10); archive_outputs()
if archive_errors: raise RuntimeError(archive_errors)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
for benchmark in ("regression", "mnist", "imagenet"):
    if (STAGE_B / "b5" / benchmark).exists(): finalize(STAGE_B, benchmark, mu_rel_default=MU_REL_DEFAULT)
print("BASELINE GATE")
print((STAGE_B / "b5_baseline_gate.json").read_text())
gates = json.loads((STAGE_B / "gate.json").read_text())
for benchmark, gate in gates.items():
    if not gate.get("measurements_valid", False):
        print("*** INVALID MEASUREMENTS:", benchmark, gate.get("baseline", gate), "***")
display(pd.DataFrame(gates).T)
for path in (STAGE_B / "b5").glob("*/bp/measurements.csv"):
    rows = pd.read_csv(path)
    benchmark = path.parents[1].name
    for (rank, mu), group in rows.groupby(["rank", "mu_rel"]):
        group.groupby("task").rho_joint.mean().plot(marker="o", label=f"r={rank}, mu={mu}")
    plt.xlabel("task"); plt.ylabel("rho_joint"); plt.title(benchmark); plt.legend()
    plt.savefig(path.parent / "rho_vs_task.png", bbox_inches="tight"); plt.show()
    display(rows[["task", "step", "site", "rank"] + [c for c in rows if c.endswith("_m") or c.endswith("_ratio")]])
archive_outputs()
